# Prediksi Dini Risiko Dropout Mahasiswa

Eksperimen untuk penelitian *Prediksi Dini Risiko Dropout Mahasiswa pada Tiga Tahap Studi Menggunakan Random Forest dengan Analisis Ketergantungan pada Atribut Keuangan*. Urutan notebook mengikuti CRISP-DM. Notebook ini berdiri sendiri dan dapat dijalankan di Kaggle (utama) atau di shell Nix laptop (cadangan).

**Data.** Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict students' dropout and academic success* [Data set]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5MC89>. Lisensi CC BY 4.0.

In [1]:
from pathlib import Path

if Path("/kaggle/working").exists():
    %pip install -q scikit-learn==1.9.0 xgboost==3.0.5 shap==0.52.0

In [2]:
import hashlib
import io
import itertools
import json
import urllib.request
import zipfile

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# NB-01: pickle scikit-learn terikat pada versinya. Nilai ini harus sama dengan baris %pip di atas.
SKLEARN_PIN = "1.9.0"
assert sklearn.__version__ == SKLEARN_PIN, (
    f"scikit-learn {sklearn.__version__} terpasang, seharusnya {SKLEARN_PIN}. "
    "Di Kaggle: restart kernel, lalu jalankan ulang dari sel pertama. Di laptop: pakai shell Nix."
)

SEED = 42
STATUSES = ["Dropout", "Graduate", "Enrolled"]
pd.set_option("display.max_rows", 200, "display.max_columns", None, "display.precision", 3)

## 1. Pemahaman masalah

Target penelitian adalah `dropout`: 1 untuk status `Dropout`, dan 0 untuk `Graduate` dan `Enrolled`. Prediksi dibuat pada tiga tahap.

| Tahap | Waktu prediksi | Atribut |
| ----- | ---------------- | ------- |
| T1 | Saat masuk | Keuangan, demografi dan keluarga, akademik saat masuk, makroekonomi |
| T2 | Akhir semester 1 | T1 ditambah 6 atribut semester 1 |
| T3 | Akhir semester 2 | T2 ditambah 6 atribut semester 2 |

Rumusan masalah:

1. Seberapa baik Random Forest memprediksi dropout saat masuk, akhir semester 1, dan akhir semester 2, dibandingkan dengan Regresi Logistik dan XGBoost?
2. Seberapa besar performa prediksi berubah pada tiap tahap jika atribut keuangan dihapus?
3. Atribut apa yang paling memengaruhi prediksi pada tiap tahap, dengan dan tanpa atribut keuangan?

Hasil penelitian ini menunjukkan pola prediktif, bukan hubungan sebab akibat.

## 2. Pengumpulan data

Notebook memakai `data/data.csv` jika ada. Jika tidak ada, seperti di Kaggle, notebook mengunduh arsip UCI. Hash `data.csv` selalu diperiksa, jadi seluruh angka di bawah berasal dari file yang sama.

In [3]:
DATA_URL = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
DATA_SHA256 = "3ef126de5cefff26eb11fbb4237f1a1401cb64b488e2f1d598c23cedeb4c45ae"

run_env = "kaggle" if Path("/kaggle/working").exists() else "laptop"
# Jupyter menjalankan kernel dari folder notebooks/, jadi root proyek dicari ke atas.
ROOT = (
    Path("/kaggle/working")
    if run_env == "kaggle"
    else next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "flake.nix").exists()), Path.cwd())
)
DATA_FILE = ROOT / "data" / "data.csv"
ARTIFACTS = ROOT / "artifacts"

if DATA_FILE.exists():
    data_bytes = DATA_FILE.read_bytes()
    data_source = "data/data.csv"
else:
    with urllib.request.urlopen(DATA_URL, timeout=120) as response:
        data_bytes = zipfile.ZipFile(io.BytesIO(response.read())).read("data.csv")
    data_source = "arsip UCI"

# DATA-01
data_sha256 = hashlib.sha256(data_bytes).hexdigest()
if data_sha256 != DATA_SHA256:
    raise ValueError(
        "Hash data.csv berbeda dari yang diharapkan, jadi notebook dihentikan.\n"
        f"  diharapkan: {DATA_SHA256}\n"
        f"  didapat:    {data_sha256}\n"
        "Ambil ulang data.csv dari arsip UCI, lalu jalankan ulang notebook."
    )

# DATA-02: file memakai BOM UTF-8, pemisah `;`, dan akhir baris CRLF. Nama kolom ke-5 diakhiri tab.
raw = pd.read_csv(io.BytesIO(data_bytes), sep=";", encoding="utf-8-sig")
raw.columns = raw.columns.str.strip()
# DATA-06: nomor baris berbasis 0 pada data.csv, bukan identitas orang. Sebagai indeks, row_id tidak ikut menjadi atribut.
raw.index.name = "row_id"

print(f"Lingkungan: {run_env}. Sumber data: {data_source}. Hash cocok: {data_sha256[:12]}...")

Lingkungan: laptop. Sumber data: data/data.csv. Hash cocok: 3ef126de5cef...


In [4]:
def status_counts(frame):
    return frame["Target"].value_counts().reindex(STATUSES, fill_value=0).to_dict()


# DATA-03: file dikunci dengan hash, jadi cabang pengisian nilai dan penghapusan duplikat tidak diperlukan.
empty_cells = int(raw.isna().sum().sum())
exact_duplicates = int(raw.duplicated().sum())
assert raw.shape == (4424, 37), raw.shape
assert empty_cells == 0, empty_cells
assert exact_duplicates == 0, exact_duplicates
assert status_counts(raw) == {"Dropout": 1421, "Graduate": 2209, "Enrolled": 794}, status_counts(raw)

print(f"File asli: {raw.shape[0]} baris, {raw.shape[1]} kolom, {empty_cells} sel kosong, {exact_duplicates} baris duplikat.")
print("Status:", status_counts(raw))

File asli: 4424 baris, 37 kolom, 0 sel kosong, 0 baris duplikat.
Status: {'Dropout': 1421, 'Graduate': 2209, 'Enrolled': 794}


## 3. Preprocessing

### Mengeluarkan `Course` 171

Seluruh baris dengan 0 mata kuliah di kedua semester berasal dari `Course` 171 (Animation and Multimedia Design), dan sebagian berlabel `Graduate`. Polanya menunjukkan kekosongan data, bukan perilaku mahasiswa, dan mempertahankannya dapat mengacaukan T2 dan T3. Karena itu, seluruh baris program ini dikeluarkan.

In [5]:
# DATA-04
is_171 = raw["Course"] == 171
no_units = (raw["Curricular units 1st sem (enrolled)"] == 0) & (raw["Curricular units 2nd sem (enrolled)"] == 0)

course_171 = {
    "name": "Animation and Multimedia Design",
    "rows": int(is_171.sum()),
    "status_counts": status_counts(raw[is_171]),
    "zero_units_both_semesters": int((is_171 & no_units).sum()),
    "zero_units_graduate": int((is_171 & no_units & (raw["Target"] == "Graduate")).sum()),
}
assert course_171["rows"] == 215, course_171
assert course_171["zero_units_both_semesters"] == 180, course_171
assert course_171["zero_units_graduate"] == 75, course_171
assert not (no_units & ~is_171).any(), "Ada baris tanpa mata kuliah di luar Course 171"

df = raw[~is_171].copy()
assert len(df) == 4209, len(df)
assert df["Course"].nunique() == 16, df["Course"].nunique()

# DATA-05: status asli tetap disimpan untuk stratifikasi dan skenario D.
df["dropout"] = (df["Target"] == "Dropout").astype(int)
assert status_counts(df) == {"Dropout": 1339, "Graduate": 2113, "Enrolled": 757}, status_counts(df)
assert (df["dropout"] == 1).sum() == 1339 and (df["dropout"] == 0).sum() == 2870
assert f"{df['dropout'].mean():.1%}" == "31.8%", df["dropout"].mean()

print(f"Course 171: {course_171}")
print(f"Setelah dikeluarkan: {len(df)} baris, {df['Course'].nunique()} program, dropout {df['dropout'].mean():.1%}.")
print("Status:", status_counts(df))

Course 171: {'name': 'Animation and Multimedia Design', 'rows': 215, 'status_counts': {'Dropout': 82, 'Graduate': 96, 'Enrolled': 37}, 'zero_units_both_semesters': 180, 'zero_units_graduate': 75}
Setelah dikeluarkan: 4209 baris, 16 program, dropout 31.8%.
Status: {'Dropout': 1339, 'Graduate': 2113, 'Enrolled': 757}


### Jenis atribut dan atribut per tahap

In [6]:
# DATA-07
FINANCIAL = ["Debtor", "Tuition fees up to date", "Scholarship holder"]
NOMINAL = [
    "Marital status",
    "Application mode",
    "Course",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]
BINARY = [
    "Daytime/evening attendance",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]
SEMESTER_FIELDS = ["credited", "enrolled", "evaluations", "approved", "grade", "without evaluations"]
SEM1 = [f"Curricular units 1st sem ({field})" for field in SEMESTER_FIELDS]
SEM2 = [f"Curricular units 2nd sem ({field})" for field in SEMESTER_FIELDS]
NUMERIC = [
    "Application order",
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
    *SEM1,
    *SEM2,
]
assert (len(NOMINAL), len(BINARY), len(NUMERIC)) == (9, 8, 19)
assert sorted(NOMINAL + BINARY + NUMERIC) == sorted(raw.columns.drop("Target")), "Daftar atribut tidak cocok dengan file"
assert df[BINARY].isin([0, 1]).all(axis=None), "Ada atribut biner dengan nilai selain 0 dan 1"

# DATA-08
T1 = [column for column in raw.columns if column not in ["Target", *SEM1, *SEM2]]
WITH_FINANCIAL = {"T1": T1, "T2": T1 + SEM1, "T3": T1 + SEM1 + SEM2}
WITHOUT_FINANCIAL = {stage: [a for a in attrs if a not in FINANCIAL] for stage, attrs in WITH_FINANCIAL.items()}

stage_attributes = {
    stage: {"with": len(WITH_FINANCIAL[stage]), "without": len(WITHOUT_FINANCIAL[stage])} for stage in WITH_FINANCIAL
}
assert stage_attributes == {
    "T1": {"with": 24, "without": 21},
    "T2": {"with": 30, "without": 27},
    "T3": {"with": 36, "without": 33},
}, stage_attributes

pd.DataFrame(stage_attributes).T.rename(columns={"with": "dengan keuangan", "without": "tanpa keuangan"})

,dengan keuangan,tanpa keuangan
T1,24,21
T2,30,27
T3,36,33


### Aturan konsistensi

Nilai `grade` 0 dipertahankan, karena selalu berarti tidak ada mata kuliah yang lulus, bukan nilai nol.

In [7]:
# DATA-09
violations = {}
grade_zero_rows = {}
for semester in ["1st", "2nd"]:
    units = lambda field: df[f"Curricular units {semester} sem ({field})"]
    violations[f"{semester}: approved > enrolled"] = int((units("approved") > units("enrolled")).sum())
    violations[f"{semester}: credited > enrolled"] = int((units("credited") > units("enrolled")).sum())
    violations[f"{semester}: evaluations < approved"] = int((units("evaluations") < units("approved")).sum())
    violations[f"{semester}: (grade = 0) berbeda dari (approved = 0)"] = int(
        ((units("grade") == 0) != (units("approved") == 0)).sum()
    )
    violations[f"{semester}: enrolled < 1"] = int((units("enrolled") < 1).sum())
    grade_zero_rows[semester] = int((units("grade") == 0).sum())

consistency_violations = sum(violations.values())
assert consistency_violations == 0, violations
assert grade_zero_rows == {"1st": 538, "2nd": 690}, grade_zero_rows

print("Baris dengan grade 0:", grade_zero_rows)
pd.Series(violations, name="pelanggaran").to_frame()

Baris dengan grade 0: {'1st': 538, '2nd': 690}


,pelanggaran
1st: approved > enrolled,0
1st: credited > enrolled,0
1st: evaluations < approved,0
1st: (grade = 0) berbeda dari (approved = 0),0
1st: enrolled < 1,0
2nd: approved > enrolled,0
2nd: credited > enrolled,0
2nd: evaluations < approved,0
2nd: (grade = 0) berbeda dari (approved = 0),0
2nd: enrolled < 1,0


### Proporsi dropout per atribut keuangan

Tabel ini memakai seluruh 4.209 baris supaya sama dengan proposal Tabel 5. Angkanya hanya menjadi deskripsi di halaman Dataset dan tidak masuk ke pemodelan.

In [8]:
def category_table(frame, attribute):
    table = pd.crosstab(frame[attribute], frame["Target"]).reindex(columns=STATUSES, fill_value=0)
    table.insert(0, "n", table.sum(axis=1))
    table["dropout_rate"] = (table["Dropout"] / table["n"]).round(6)
    table = table.rename_axis(index="category", columns=None).reset_index()
    table.insert(0, "attribute", attribute)
    return table


financial_rates = pd.concat([category_table(df, attribute) for attribute in FINANCIAL], ignore_index=True)

financial_percent = {
    (row.attribute, row.category): round(row.dropout_rate * 100, 1) for row in financial_rates.itertuples()
}
assert financial_percent == {
    ("Debtor", 0): 27.9,
    ("Debtor", 1): 62.4,
    ("Tuition fees up to date", 0): 87.2,
    ("Tuition fees up to date", 1): 24.4,
    ("Scholarship holder", 0): 38.5,
    ("Scholarship holder", 1): 11.7,
}, financial_percent

financial_rates

,attribute,category,n,Dropout,Graduate,Enrolled,dropout_rate
0,Debtor,0,3735,1043,2017,675,0.279
1,Debtor,1,474,296,96,82,0.624
2,Tuition fees up to date,0,499,435,23,41,0.872
3,Tuition fees up to date,1,3710,904,2090,716,0.244
4,Scholarship holder,0,3162,1217,1312,633,0.385
5,Scholarship holder,1,1047,122,801,124,0.117


### Pembagian data

Pembagian 80:20 distratifikasi menurut tiga status. Pembagian dilakukan sebelum eksplorasi, supaya eksplorasi hanya melihat data latih. Baris uji yang sama dipakai di semua skenario, dan skenario D hanya mengeluarkan `Enrolled` dari kedua himpunan.

In [9]:
# DATA-10
train, test = train_test_split(df, test_size=0.2, stratify=df["Target"], random_state=SEED)
train_d = train[train["Target"] != "Enrolled"]
test_d = test[test["Target"] != "Enrolled"]

split_table = pd.DataFrame(
    {
        "Semua": status_counts(df),
        "Latih": status_counts(train),
        "Uji": status_counts(test),
        "Latih skenario D": status_counts(train_d),
        "Uji skenario D": status_counts(test_d),
    }
).T
split_table.insert(0, "Jumlah", split_table.sum(axis=1))

assert split_table.to_dict("index") == {
    "Semua": {"Jumlah": 4209, "Dropout": 1339, "Graduate": 2113, "Enrolled": 757},
    "Latih": {"Jumlah": 3367, "Dropout": 1071, "Graduate": 1690, "Enrolled": 606},
    "Uji": {"Jumlah": 842, "Dropout": 268, "Graduate": 423, "Enrolled": 151},
    "Latih skenario D": {"Jumlah": 2761, "Dropout": 1071, "Graduate": 1690, "Enrolled": 0},
    "Uji skenario D": {"Jumlah": 691, "Dropout": 268, "Graduate": 423, "Enrolled": 0},
}, split_table
assert train.index.intersection(test.index).empty, "Ada row_id yang masuk data latih dan data uji"

split_table

,Jumlah,Dropout,Graduate,Enrolled
Semua,4209,1339,2113,757
Latih,3367,1071,1690,606
Uji,842,268,423,151
Latih skenario D,2761,1071,1690,0
Uji skenario D,691,268,423,0


### One-hot encoding dan penskalaan

Encoder berada di dalam `Pipeline`, sehingga dihitung ulang di tiap fold dan hanya dari data latih. Atribut nominal memakai one-hot, dan kategori di bawah 1% digabung menjadi satu kolom. Atribut numerik hanya diskalakan untuk Regresi Logistik, dan atribut biner tetap 0 dan 1.

In [10]:
def make_encoder(attributes, scale_numeric):
    # NB-05: model pohon memakai scale_numeric=False, Regresi Logistik memakai scale_numeric=True.
    onehot = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01, sparse_output=False)
    transformers = [("nominal", onehot, NOMINAL)]
    if scale_numeric:
        transformers.append(("numeric", StandardScaler(), [a for a in attributes if a in NUMERIC]))
    return ColumnTransformer(transformers, remainder="passthrough")

In [11]:
# set_output hanya dipakai di sel ini, supaya kolom hasil encoder dapat diperiksa menurut nama.
X_t3 = train[WITH_FINANCIAL["T3"]]
tree_encoder = make_encoder(WITH_FINANCIAL["T3"], scale_numeric=False).set_output(transform="pandas").fit(X_t3)
linear_encoder = make_encoder(WITH_FINANCIAL["T3"], scale_numeric=True).set_output(transform="pandas").fit(X_t3)
tree_out, linear_out = tree_encoder.transform(X_t3), linear_encoder.transform(X_t3)

onehot_rows = []
for attribute in NOMINAL:
    columns = [column for column in tree_out.columns if column.startswith(f"nominal__{attribute}_")]
    counts = train[attribute].value_counts()
    frequent = int((counts >= 0.01 * len(train)).sum())
    assert len(columns) == frequent + (frequent < len(counts)), f"{attribute}: jumlah kolom one-hot tidak sesuai min_frequency"
    assert (tree_out[columns].sum(axis=1) == 1).all(), f"{attribute}: tiap baris harus tepat satu kolom one-hot"
    onehot_rows.append({"attribute": attribute, "kategori_di_data_latih": len(counts), "kolom_one_hot": len(columns)})

tree_numeric = tree_out[[f"remainder__{attribute}" for attribute in NUMERIC]]
assert (tree_numeric.to_numpy() == train[NUMERIC].to_numpy()).all(), "Model pohon tidak boleh menskalakan atribut numerik"
linear_binary = linear_out[[f"remainder__{attribute}" for attribute in BINARY]]
assert (linear_binary.to_numpy() == train[BINARY].to_numpy()).all(), "Atribut biner tidak boleh diskalakan"
linear_numeric = linear_out[[f"numeric__{attribute}" for attribute in NUMERIC]]
assert (linear_numeric.mean().abs() < 1e-9).all(), "Atribut numerik Regresi Logistik harus berpusat di 0"
assert ((linear_numeric.std(ddof=0) - 1).abs() < 1e-9).all(), "Atribut numerik Regresi Logistik harus bersimpangan baku 1"

unseen = tree_encoder.transform(X_t3.head(1).assign(Course=-1))
assert unseen["nominal__Course_infrequent_sklearn"].iloc[0] == 1, "Kategori baru harus masuk ke kolom kategori jarang"

pd.DataFrame(onehot_rows)

,attribute,kategori_di_data_latih,kolom_one_hot
0,Marital status,6,4
1,Application mode,18,10
2,Course,16,16
3,Previous qualification,17,6
4,Nacionality,17,2
5,Mother's qualification,28,9
6,Father's qualification,31,8
7,Mother's occupation,31,11
8,Father's occupation,40,13


## 4. Eksplorasi data

Seluruh tabel pada bagian ini hanya memakai `train`. Data uji tidak dipakai, supaya tidak memengaruhi rancangan model.

### Distribusi atribut numerik per status

In [12]:
numeric_by_status = (
    train.melt(id_vars="Target", value_vars=NUMERIC, var_name="attribute")
    .groupby(["attribute", "Target"])["value"]
    .describe()[["mean", "25%", "50%", "75%"]]
    .rename(columns={"25%": "q1", "50%": "median", "75%": "q3"})
    .round(6)
    .rename_axis(index=["attribute", "status"])
    .reset_index()
)

numeric_by_status.pivot(index="attribute", columns="status", values=["q1", "median", "q3"]).reindex(
    index=NUMERIC, columns=pd.MultiIndex.from_product([["q1", "median", "q3"], STATUSES])
)

q1                    \
                                               Dropout Graduate Enrolled   
attribute                                                                  
Application order                                  1.0    1.000      1.0   
Previous qualification (grade)                   122.5  125.000    122.0   
Admission grade                                  115.0  119.400    116.5   
Age at enrollment                                 19.0   18.000     19.0   
Unemployment rate                                  9.4    9.400      9.4   
Inflation rate                                     0.3    0.300      0.3   
GDP                                               -1.7   -1.700     -1.7   
Curricular units 1st sem (credited)                0.0    0.000      0.0   
Curricular units 1st sem (enrolled)                5.0    6.000      5.0   
Curricular units 1st sem (evaluations)             6.0    7.000      8.0   
Curricular units 1st sem (approved)                0.0    5.000      3.0   
Curricular units 1st sem (grade)                   0.0   12.200     11.0   
Curricular units 1st sem (without evaluations)     0.0    0.000      0.0   
Curricular units 2nd sem (credited)                0.0    0.000      0.0   
Curricular units 2nd sem (enrolled)                5.0    6.000      5.0   
Curricular units 2nd sem (evaluations)             5.0    6.000      8.0   
Curricular units 2nd sem (approved)                0.0    5.000      3.0   
Curricular units 2nd sem (grade)                   0.0   12.232     11.0   
Curricular units 2nd sem (without evaluations)     0.0    0.000      0.0   

                                                median                    \
                                               Dropout Graduate Enrolled   
attribute                                                                  
Application order                                 1.00    1.000     1.00   
Previous qualification (grade)                  133.10  133.100   130.00   
Admission grade                                 123.60  127.200   123.50   
Age at enrollment                                24.00   19.000    20.00   
Unemployment rate                                11.10   11.100    11.10   
Inflation rate                                    1.40    0.600     1.40   
GDP                                               0.32    0.790     0.32   
Curricular units 1st sem (credited)               0.00    0.000     0.00   
Curricular units 1st sem (enrolled)               6.00    6.000     6.00   
Curricular units 1st sem (evaluations)            8.00    8.000     9.00   
Curricular units 1st sem (approved)               2.00    6.000     5.00   
Curricular units 1st sem (grade)                 11.00   13.059    12.00   
Curricular units 1st sem (without evaluations)    0.00    0.000     0.00   
Curricular units 2nd sem (credited)               0.00    0.000     0.00   
Curricular units 2nd sem (enrolled)               6.00    6.000     6.00   
Curricular units 2nd sem (evaluations)            8.00    8.000     9.00   
Curricular units 2nd sem (approved)               1.00    6.000     4.00   
Curricular units 2nd sem (grade)                 10.00   13.035    12.00   
Curricular units 2nd sem (without evaluations)    0.00    0.000     0.00   

                                                    q3                    
                                               Dropout Graduate Enrolled  
attribute                                                                 
Application order                                 2.00    2.000    2.000  
Previous qualification (grade)                  137.50  141.000  140.000  
Admission grade                                 132.95  135.900  132.400  
Age at enrollment                                31.00   21.000   24.000  
Unemployment rate                                13.90   13.900   12.700  
Inflation rate                                    2.60    2.600    2.600  
GDP                            

### Proporsi dropout per kategori

Tabel lengkap untuk semua kategori disimpan di `dataset_summary.json`. Kode kategori mengikuti tabel variabel UCI. Kategori kecil menghasilkan proporsi yang tidak stabil, jadi ringkasan atribut nominal hanya memakai kategori dengan minimal 1% baris latih.

In [13]:
category_rates = pd.concat([category_table(train, attribute) for attribute in NOMINAL + BINARY], ignore_index=True)

binary_rates = category_rates[category_rates["attribute"].isin(BINARY)]
binary_rates.pivot(index="attribute", columns="category", values=["n", "dropout_rate"]).reindex(BINARY).astype(
    {("n", 0): int, ("n", 1): int}
)

n       dropout_rate       
category                       0     1            0      1
attribute                                                 
Daytime/evening attendance   394  2973        0.439  0.302
Displaced                   1577  1790        0.370  0.272
Educational special needs   3330    37        0.318  0.324
Debtor                      2993   374        0.279  0.628
Tuition fees up to date      390  2977        0.872  0.246
Gender                      2219  1148        0.251  0.447
Scholarship holder          2518   849        0.386  0.118
International               3290    77        0.319  0.286

In [14]:
nominal_rates = category_rates[category_rates["attribute"].isin(NOMINAL)]
frequent_rates = nominal_rates[nominal_rates["n"] >= 0.01 * len(train)]

nominal_overview = frequent_rates.groupby("attribute").agg(
    kategori_minimal_1_persen=("category", "size"),
    dropout_terendah=("dropout_rate", "min"),
    dropout_tertinggi=("dropout_rate", "max"),
)
nominal_overview.insert(0, "kategori_di_data_latih", nominal_rates.groupby("attribute").size())
nominal_overview.reindex(NOMINAL)

,kategori_di_data_latih,kategori_minimal_1_persen,dropout_terendah,dropout_tertinggi
attribute,,,,
Marital status,6,3,0.297,0.474
Application mode,18,9,0.189,0.630
Course,16,15,0.149,0.596
Previous qualification,17,5,0.281,0.662
Nacionality,17,1,0.319,0.319
Mother's qualification,28,8,0.146,0.755
Father's qualification,31,7,0.225,0.734
Mother's occupation,31,10,0.260,0.768
Father's occupation,40,12,0.255,0.765


In [15]:
course_rates = frequent_rates[frequent_rates["attribute"] == "Course"].drop(columns="attribute")
course_rates.sort_values("dropout_rate", ascending=False, kind="stable").reset_index(drop=True)

,category,n,Dropout,Graduate,Enrolled,dropout_rate
0,9130,114,68,32,14,0.596
1,9119,135,70,10,55,0.519
2,9991,222,113,61,48,0.509
3,9853,158,67,51,40,0.424
4,9254,186,74,80,32,0.398
5,9003,163,64,70,29,0.393
6,9556,72,27,32,13,0.375
7,9670,226,82,105,39,0.363
8,8014,172,60,97,15,0.349
9,9147,296,102,105,89,0.345


### Korelasi antaratribut

Korelasi Pearson hanya dihitung pada atribut numerik dan biner, karena kode atribut nominal tidak memiliki urutan.

In [16]:
corr = train[NUMERIC + BINARY].corr()
correlations = pd.DataFrame(
    [(a, b, corr.loc[a, b]) for a, b in itertools.combinations(corr.columns, 2)],
    columns=["attribute_a", "attribute_b", "r"],
)
assert correlations["r"].notna().all(), "Ada atribut tanpa variasi pada data latih"

top_correlations = (
    correlations.sort_values("r", key=abs, ascending=False, kind="stable").head(20).round(6).reset_index(drop=True)
)
top_correlations

,attribute_a,attribute_b,r
0,Curricular units 1st sem (credited),Curricular units 2nd sem (credited),0.942
1,Curricular units 1st sem (enrolled),Curricular units 2nd sem (enrolled),0.914
2,Curricular units 1st sem (approved),Curricular units 2nd sem (approved),0.891
3,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),0.860
4,Curricular units 1st sem (enrolled),Curricular units 2nd sem (credited),0.841
5,Curricular units 1st sem (grade),Curricular units 2nd sem (grade),0.807
6,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),0.773
7,Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),0.746
8,Curricular units 1st sem (evaluations),Curricular units 2nd sem (evaluations),0.738
9,Curricular units 1st sem (enrolled),Curricular units 1st sem (approved),0.734


In [17]:
first_stage = {attribute: "T1" for attribute in T1} | {a: "T2" for a in SEM1} | {a: "T3" for a in SEM2}

target_correlations = (
    train[NUMERIC + BINARY]
    .corrwith(train["dropout"])
    .rename("r")
    .rename_axis("attribute")
    .reset_index()
    .assign(stage=lambda table: table["attribute"].map(first_stage))
    .sort_values("r", key=abs, ascending=False, kind="stable")
    .round(6)
    .reset_index(drop=True)
)
target_correlations

,attribute,r,stage
0,Curricular units 2nd sem (grade),-0.626,T3
1,Curricular units 2nd sem (approved),-0.612,T3
2,Curricular units 1st sem (grade),-0.533,T2
3,Curricular units 1st sem (approved),-0.513,T2
4,Tuition fees up to date,-0.430,T1
5,Age at enrollment,0.269,T1
6,Scholarship holder,-0.250,T1
7,Debtor,0.235,T1
8,Gender,0.199,T1
9,Curricular units 2nd sem (evaluations),-0.162,T3


### Ringkasan eksplorasi

In [18]:
def strongest(stage, k=3):
    rows = target_correlations[target_correlations["stage"] == stage].head(k)
    return ", ".join(f"{row.attribute} ({row.r:+.2f})" for row in rows.itertuples())


high_pairs = correlations[correlations["r"].abs() >= 0.7]
pairs_across_semesters = high_pairs[
    high_pairs["attribute_a"].isin(SEM1) & high_pairs["attribute_b"].isin(SEM2)
]
rate_by_course = course_rates.set_index("category")["dropout_rate"]

print(f"Data latih: {len(train)} baris, dropout {train['dropout'].mean():.1%}.")
for stage in ["T1", "T2", "T3"]:
    print(f"Korelasi terkuat dengan dropout, atribut baru pada {stage}: {strongest(stage)}.")
for semester in ["1st", "2nd"]:
    grade = f"Curricular units {semester} sem (grade)"
    passed = train[train[grade] > 0]
    print(
        f"{grade}: r dengan dropout {train[grade].corr(train['dropout']):+.2f}, "
        f"tanpa baris grade 0 {passed[grade].corr(passed['dropout']):+.2f}. "
        f"Dropout {train.loc[train[grade] == 0, 'dropout'].mean():.1%} pada grade 0, "
        f"{passed['dropout'].mean():.1%} pada grade di atas 0."
    )
print(
    f"Pasangan atribut dengan |r| >= 0.7: {len(high_pairs)}, "
    f"{len(pairs_across_semesters)} di antaranya antara atribut semester 1 dan semester 2."
)
print(
    f"Proporsi dropout per Course: tertinggi {rate_by_course.max():.1%} (kode {rate_by_course.idxmax()}), "
    f"terendah {rate_by_course.min():.1%} (kode {rate_by_course.idxmin()})."
)

Data latih: 3367 baris, dropout 31.8%.
Korelasi terkuat dengan dropout, atribut baru pada T1: Tuition fees up to date (-0.43), Age at enrollment (+0.27), Scholarship holder (-0.25).
Korelasi terkuat dengan dropout, atribut baru pada T2: Curricular units 1st sem (grade) (-0.53), Curricular units 1st sem (approved) (-0.51), Curricular units 1st sem (enrolled) (-0.13).
Korelasi terkuat dengan dropout, atribut baru pada T3: Curricular units 2nd sem (grade) (-0.63), Curricular units 2nd sem (approved) (-0.61), Curricular units 2nd sem (evaluations) (-0.16).
Curricular units 1st sem (grade): r dengan dropout -0.53, tanpa baris grade 0 -0.23. Dropout 91.1% pada grade 0, 22.7% pada grade di atas 0.
Curricular units 2nd sem (grade): r dengan dropout -0.63, tanpa baris grade 0 -0.22. Dropout 94.0% pada grade 0, 19.2% pada grade di atas 0.
Pasangan atribut dengan |r| >= 0.7: 11, 7 di antaranya antara atribut semester 1 dan semester 2.
Proporsi dropout per Course: tertinggi 59.6% (kode 9130), tere

**Catatan penafsiran**

- Angka di atas adalah korelasi linear satu atribut pada data latih. Angka ini bukan perkiraan performa model dan bukan hubungan sebab akibat.
- Pada T1, korelasi terkuat dimiliki `Tuition fees up to date`, yaitu atribut keuangan. Waktu pencatatannya tidak dirinci pada sumber data, sehingga rumusan masalah 2 perlu mengukur seberapa jauh T1 bergantung pada atribut ini.
- Sebagian besar korelasi `grade` berasal dari baris `grade` 0, yaitu mahasiswa tanpa mata kuliah yang lulus, dan hampir semua baris ini berakhir dropout. File publik tidak memiliki tanggal keluar, jadi nilai 0 bisa saja mencatat mahasiswa yang sudah berhenti kuliah. Hal ini sejalan dengan keterbatasan rancangan bahwa T3 mungkin terlalu optimistis.
- Atribut semester 1 dan semester 2 saling berkorelasi kuat. Pada T3, kontribusi SHAP dapat terbagi di antara atribut yang serupa, jadi peringkat atribut semester 1 pada T3 perlu dibaca hati-hati. Multikolinearitas ini juga menjadi alasan Regresi Logistik memakai regularisasi L2.
- Banyak kategori nominal berada di bawah 1% dan akan digabung saat one-hot encoding. `Nacionality` hanya menyisakan satu kategori di atas batas itu.

Eksplorasi ini tidak mengubah rancangan. Seluruh 36 atribut, ruang pencarian, dan ambang keputusan tetap mengikuti proposal.

### Simpan ringkasan

In [19]:
dataset_summary = {
    "status_counts": {"file": status_counts(raw), "after_exclusion": status_counts(df)},
    "course_171": course_171,
    "quality": {
        "rows": raw.shape[0],
        "columns": raw.shape[1],
        "empty_cells": empty_cells,
        "exact_duplicates": exact_duplicates,
        "rows_after_exclusion": len(df),
        "programs_after_exclusion": int(df["Course"].nunique()),
        "grade_zero_rows": grade_zero_rows,
        "consistency_violations": consistency_violations,
    },
    "stage_attributes": stage_attributes,
    "financial_dropout_rates": {"source": "all", "n_rows": len(df), "rows": financial_rates.to_dict("records")},
    "exploration": {
        "source": "train",
        "n_rows": len(train),
        "dropout_rate": round(float(train["dropout"].mean()), 6),
        "numeric_by_status": numeric_by_status.to_dict("records"),
        "category_rates": category_rates.to_dict("records"),
        "top_correlations": top_correlations.to_dict("records"),
        "target_correlations": target_correlations.to_dict("records"),
    },
}

ARTIFACTS.mkdir(parents=True, exist_ok=True)
summary_text = json.dumps(dataset_summary, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
(ARTIFACTS / "dataset_summary.json").write_text(summary_text, encoding="utf-8")
print(f"artifacts/dataset_summary.json ditulis, {len(summary_text) / 1024:.0f} KB.")

artifacts/dataset_summary.json ditulis, 62 KB.


## 5. Pemodelan

### Baseline Regresi Logistik

Baseline memakai atribut skenario A pada tiap tahap, dengan konfigurasi dan pencarian parameter yang sama dengan model akhir. Seluruh skor di bawah adalah PR-AUC dari 5-fold CV pada data latih. Data uji belum dinilai, jadi skor ini bukan hasil penelitian.

In [20]:
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)


def tune(classifier, attributes, scale_numeric, space, n_iter):
    # NB-07: pencarian parameter hanya melihat data latih.
    pipeline = Pipeline([("enc", make_encoder(attributes, scale_numeric)), ("clf", classifier)])
    search = RandomizedSearchCV(
        pipeline, space, n_iter=n_iter, scoring="average_precision", n_jobs=-1, cv=CV, random_state=SEED
    )
    return search.fit(train[attributes], train["dropout"])


# NB-06: argumen penalty tidak diisi, karena L2 sudah menjadi bawaan.
LR_SPACE = {"clf__C": [0.01, 0.1, 1, 10]}
searches = {
    f"A_{stage}_lr": tune(LogisticRegression(class_weight="balanced", max_iter=5000), attributes, True, LR_SPACE, 4)
    for stage, attributes in WITH_FINANCIAL.items()
}

for stage, attributes in WITH_FINANCIAL.items():
    search = searches[f"A_{stage}_lr"]
    model = search.best_estimator_
    assert list(model.feature_names_in_) == attributes, f"{stage}: atribut masukan tidak sesuai tahap"
    assert sorted(search.cv_results_["param_clf__C"].tolist()) == LR_SPACE["clf__C"], f"{stage}: tidak semua nilai C dicoba"
    assert search.n_splits_ == 5, search.n_splits_
    assert model["clf"].n_iter_.max() < model["clf"].max_iter, f"{stage}: Regresi Logistik belum konvergen"

print(f"Patokan terendah PR-AUC pada data latih: {train['dropout'].mean():.3f}.")
pd.DataFrame(
    [
        {
            "model_id": model_id,
            "atribut": search.n_features_in_,
            "kolom_setelah_encoding": len(search.best_estimator_["enc"].get_feature_names_out()),
            "C_terbaik": search.best_params_["clf__C"],
            "cv_pr_auc": round(float(search.best_score_), 6),
            "std_antarfold": round(float(search.cv_results_["std_test_score"][search.best_index_]), 6),
        }
        for model_id, search in searches.items()
    ]
)

Patokan terendah PR-AUC pada data latih: 0.318.


,model_id,atribut,kolom_setelah_encoding,C_terbaik,cv_pr_auc,std_antarfold
0,A_T1_lr,24,94,1,0.739,0.018
1,A_T2_lr,30,100,1,0.869,0.005
2,A_T3_lr,36,106,1,0.894,0.006


In [21]:
# Skor rata-rata tiap nilai C, untuk melihat kepekaan baseline terhadap parameter.
pd.DataFrame(
    {
        model_id: pd.Series(search.cv_results_["mean_test_score"], index=search.cv_results_["param_clf__C"].tolist())
        for model_id, search in searches.items()
    }
).rename_axis("C").sort_index()

,A_T1_lr,A_T2_lr,A_T3_lr
C,,,
0.01,0.700,0.845,0.875
0.10,0.737,0.866,0.893
1.00,0.739,0.869,0.894
10.00,0.738,0.867,0.892


**Catatan penafsiran.** Pada ketiga tahap, skor baseline jauh di atas patokan terendah dan naik dari T1 ke T3, dengan kenaikan terbesar dari T1 ke T2. Perbandingan antartahap dan antaralgoritma baru dinilai pada data uji, setelah seluruh parameter skenario ditetapkan. Skor T3 tetap perlu dibaca dengan keterbatasan bahwa tahap ini mungkin terlalu optimistis.